# 05 — Bayesian Volatility and Downside Risk

## 1. Title and objective

**Objective.** Estimate stock-level volatility and downside loss probabilities from the `daily_returns` SQL table using Bayesian models.

This notebook focuses on two practical risk questions:

1. **How volatile is each stock after accounting for fat-tailed daily returns?**
2. **How likely is each stock to experience large negative daily returns?**

Volatility and downside risk are related, but they answer different finance questions. Volatility measures typical variation around the expected return, while downside-risk models estimate the probability of crossing a specific loss threshold such as **-2%** or **-5%** in one trading day.

## Setup: imports, paths, and reproducibility settings

The notebook uses the same reusable PyMC helpers introduced in Notebook 4. The path setup allows execution from either the repository root or the `notebooks/` directory, and generated figures are saved under `reports/figures`.

In [ ]:
from pathlib import Path
import sys

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

# Resolve the repository root whether this notebook is launched from the repo
# root or from the notebooks/ directory.
NOTEBOOK_PATH = Path.cwd().resolve()
ROOT = NOTEBOOK_PATH if (NOTEBOOK_PATH / "src").exists() else NOTEBOOK_PATH.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.bayesian_models import (
    build_bayesian_bad_day_model,
    build_hierarchical_student_t_return_model,
    posterior_bad_day_summary,
    posterior_probability_highest_bad_day_risk,
    prepare_downside_data,
    prepare_returns_for_bayesian_model,
    sample_model,
)
from src.config import (
    DUCKDB_PATH,
    FIGURES_DIR,
    RAW_CSV_PATH,
    TRADING_DAYS_PER_YEAR,
)
from src.model_io import (
    POSTERIOR_SAMPLES_DIR,
    TABLES_DIR,
    ensure_model_dirs,
    load_inference_data,
    save_inference_data,
    save_summary_table,
)
from src.sql_utils import initialize_database, query_to_df, run_sql_pipeline

RANDOM_SEED = 42
DRAWS = 1000
TUNE = 1000
CHAINS = 4
TARGET_ACCEPT = 0.90

# If True, overwrite cached posterior samples. If False, reuse cached NetCDF
# samples when available and refit only when the cache is missing.
REFIT_MODELS = False

RAW_CSV_FOR_NOTEBOOK = RAW_CSV_PATH if RAW_CSV_PATH.exists() else ROOT / "tech_stocks.csv"
POSTERIOR_DIR = POSTERIOR_SAMPLES_DIR
ensure_model_dirs()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
az.style.use("arviz-whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.6f}".format)

rng = np.random.default_rng(RANDOM_SEED)

print(f"Repository root: {ROOT}")
print(f"Raw CSV used:     {RAW_CSV_FOR_NOTEBOOK}")
print(f"DuckDB path:      {DUCKDB_PATH}")
print(f"Figures dir:      {FIGURES_DIR}")
print(f"Posterior dir:    {POSTERIOR_DIR}")

## 2. Load `daily_returns` from DuckDB

The SQL pipeline produces the `daily_returns` table used throughout the Bayesian risk analysis. This notebook requires valid `simple_return` and `log_return` observations because volatility is modeled with log returns and downside events are defined with simple returns.

## Risk-metric methodology

The risk sections separate modeling returns from intuitive loss reporting:

- Volatility is modeled with **log returns** and annualized as `daily log-return volatility × sqrt(252)`.
- Downside events use **simple returns** when thresholds are stated as intuitive losses such as `-2%` or `-5%`.
- Drawdown uses the adjusted-close path: running peak is the highest adjusted close to date, current drawdown is `adj_close / running_peak - 1`, and maximum drawdown is the worst observed drawdown.
- Historical 5% VaR is the 5th percentile of daily returns. It is a loss threshold, not the average loss. Historical 5% CVaR is the average return conditional on being in the worst 5% of days, so it better summarizes severe tail outcomes.
- Tail-risk models use Student-t likelihoods because a normal likelihood tends to understate the frequency and magnitude of extreme daily returns.



In [ ]:
con = initialize_database(raw_csv_path=RAW_CSV_FOR_NOTEBOOK, db_path=DUCKDB_PATH)
run_sql_pipeline(con)

daily_returns = query_to_df(
    con,
    """
    SELECT
        symbol,
        date,
        adj_close,
        volume,
        simple_return,
        log_return,
        dollar_volume
    FROM daily_returns
    WHERE simple_return IS NOT NULL
      AND log_return IS NOT NULL
    ORDER BY symbol, date
    """,
)

display(daily_returns.head())
print(
    f"Loaded {len(daily_returns):,} daily return rows across "
    f"{daily_returns['symbol'].nunique()} stocks."
)

In [ ]:
raw_risk_summary = (
    daily_returns.groupby("symbol")
    .agg(
        observations=("simple_return", "size"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        raw_daily_log_return_volatility=("log_return", "std"),
        raw_bad_day_frequency_2pct=("simple_return", lambda x: (x <= -0.02).mean()),
        raw_extreme_loss_frequency_5pct=("simple_return", lambda x: (x <= -0.05).mean()),
    )
    .reset_index()
)
raw_risk_summary["raw_annualized_volatility"] = (
    raw_risk_summary["raw_daily_log_return_volatility"]
    * np.sqrt(TRADING_DAYS_PER_YEAR)
)
raw_risk_summary["raw_bad_days_per_year_2pct"] = (
    raw_risk_summary["raw_bad_day_frequency_2pct"] * TRADING_DAYS_PER_YEAR
)
raw_risk_summary["raw_extreme_loss_days_per_year_5pct"] = (
    raw_risk_summary["raw_extreme_loss_frequency_5pct"] * TRADING_DAYS_PER_YEAR
)

display(raw_risk_summary)

## 3. Part A: Volatility from the hierarchical Student-t return model

Notebook 4 introduced a hierarchical Student-t return model for daily log returns. The same model is useful here because it estimates a stock-specific `stock_sigma` while allowing fat-tailed shocks through the Student-t likelihood.

Finance interpretation:

- `stock_sigma` is the posterior estimate of each stock's **daily log-return volatility**.
- Multiplying daily sigma by $\sqrt{252}$ converts it to an approximate **annualized volatility**.
- The posterior distribution gives a credible interval, not just a single historical estimate.

In [ ]:
returns, stock_idx, symbol_lookup, volatility_modeling_df = prepare_returns_for_bayesian_model(
    daily_returns
)
symbols = [symbol_lookup[idx] for idx in sorted(symbol_lookup)]
n_stocks = len(symbols)

student_t_cache_path = POSTERIOR_DIR / "return_model.nc"
student_t_model = build_hierarchical_student_t_return_model(
    returns=returns,
    stock_idx=stock_idx,
    n_stocks=n_stocks,
)

if student_t_cache_path.exists() and not REFIT_MODELS:
    student_t_idata = load_inference_data(student_t_cache_path)
    print(f"Loaded cached Student-t posterior samples from {student_t_cache_path}")
else:
    student_t_idata = sample_model(
        student_t_model,
        draws=DRAWS,
        tune=TUNE,
        chains=CHAINS,
        target_accept=TARGET_ACCEPT,
    )
    save_inference_data(student_t_idata, student_t_cache_path)
    print(f"Saved Student-t posterior samples to {student_t_cache_path}")

student_t_idata

### Posterior annualized volatility table with 90% credible intervals

The table below extracts posterior `stock_sigma`, annualizes each posterior draw by multiplying by $\sqrt{252}$, and reports posterior means plus 5% and 95% quantiles.

In [ ]:
stock_sigma_samples = (
    student_t_idata.posterior["stock_sigma"]
    .assign_coords(stock=symbols)
    .stack(sample=("chain", "draw"))
    .transpose("stock", "sample")
    .values
)
annualized_vol_samples = stock_sigma_samples * np.sqrt(TRADING_DAYS_PER_YEAR)
annualized_vol_quantiles = np.quantile(annualized_vol_samples, [0.05, 0.50, 0.95], axis=1)

volatility_summary = pd.DataFrame(
    {
        "symbol": symbols,
        "posterior_annualized_volatility_mean": annualized_vol_samples.mean(axis=1),
        "annualized_volatility_ci_5": annualized_vol_quantiles[0],
        "annualized_volatility_median": annualized_vol_quantiles[1],
        "annualized_volatility_ci_95": annualized_vol_quantiles[2],
    }
).sort_values("posterior_annualized_volatility_mean", ascending=False)

styled_volatility_summary = volatility_summary.style.format(
    {
        "posterior_annualized_volatility_mean": "{:.2%}",
        "annualized_volatility_ci_5": "{:.2%}",
        "annualized_volatility_median": "{:.2%}",
        "annualized_volatility_ci_95": "{:.2%}",
    }
)

volatility_summary_path = save_summary_table(
    volatility_summary,
    TABLES_DIR / "05_volatility_summary.csv",
)
print(f"Saved volatility summary to {volatility_summary_path}")
display(styled_volatility_summary)

### Plot posterior volatility by symbol

The interval plot shows posterior uncertainty around each stock's annualized volatility. Wider intervals indicate less precise volatility estimates, usually because of heavier tails, outliers, or limited effective information.

In [ ]:
volatility_plot_df = volatility_summary.sort_values(
    "posterior_annualized_volatility_mean", ascending=True
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.errorbar(
    volatility_plot_df["posterior_annualized_volatility_mean"],
    volatility_plot_df["symbol"],
    xerr=[
        volatility_plot_df["posterior_annualized_volatility_mean"]
        - volatility_plot_df["annualized_volatility_ci_5"],
        volatility_plot_df["annualized_volatility_ci_95"]
        - volatility_plot_df["posterior_annualized_volatility_mean"],
    ],
    fmt="o",
    color="#4C78A8",
    ecolor="#4C78A8",
    elinewidth=2,
    capsize=4,
)
ax.set_title("Posterior annualized volatility by stock")
ax.set_xlabel("Annualized volatility")
ax.set_ylabel("Symbol")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
fig.tight_layout()
volatility_plot_path = FIGURES_DIR / "05_posterior_annualized_volatility_by_symbol.png"
fig.savefig(volatility_plot_path, dpi=300, bbox_inches="tight")
volatility_plot_path

### Probability each stock has the highest volatility

For each posterior draw, the stock with the largest annualized volatility is counted. The resulting share of draws is the posterior probability that a stock is the most volatile in the comparison set.

In [ ]:
highest_volatility_idx = np.argmax(annualized_vol_samples, axis=0)
highest_volatility_probabilities = (
    np.bincount(highest_volatility_idx, minlength=n_stocks)
    / highest_volatility_idx.size
)
highest_volatility_table = pd.DataFrame(
    {
        "symbol": symbols,
        "probability_highest_volatility": highest_volatility_probabilities,
    }
).sort_values("probability_highest_volatility", ascending=False)

display(
    highest_volatility_table.style.format(
        {"probability_highest_volatility": "{:.1%}"}
    )
)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(
    data=highest_volatility_table.sort_values("probability_highest_volatility"),
    x="probability_highest_volatility",
    y="symbol",
    color="#F58518",
    ax=ax,
)
ax.set_xlim(0, max(1.0, highest_volatility_table["probability_highest_volatility"].max() * 1.1))
ax.set_title("Posterior probability of highest volatility")
ax.set_xlabel("Pr(highest annualized volatility)")
ax.set_ylabel("Symbol")
fig.tight_layout()
highest_volatility_path = FIGURES_DIR / "05_probability_highest_volatility.png"
fig.savefig(highest_volatility_path, dpi=300, bbox_inches="tight")
highest_volatility_path

## 4. Part B: Bayesian bad-day model

A **bad day** is defined as:

$$\text{simple_return} \leq -2\%$$

Downside risk is different from volatility because it focuses on the probability of a specific negative outcome. A stock can have high volatility because it has both large positive and large negative moves, while downside risk asks how often losses breach a practical pain threshold. For risk management, threshold probabilities can be easier to connect to stop-loss rules, drawdown concerns, and investor behavior than a symmetric volatility number.

In [ ]:
bad_day_threshold = -0.02
bad_day_y, bad_day_stock_idx, bad_day_symbol_lookup, bad_day_modeling_df = prepare_downside_data(
    daily_returns,
    threshold=bad_day_threshold,
)
bad_day_symbols = [bad_day_symbol_lookup[idx] for idx in sorted(bad_day_symbol_lookup)]

bad_day_cache_path = POSTERIOR_DIR / "downside_model_2pct.nc"
bad_day_model = build_bayesian_bad_day_model(
    y=bad_day_y,
    stock_idx=bad_day_stock_idx,
    n_stocks=len(bad_day_symbols),
)

if bad_day_cache_path.exists() and not REFIT_MODELS:
    bad_day_idata = load_inference_data(bad_day_cache_path)
    print(f"Loaded cached -2% bad-day posterior samples from {bad_day_cache_path}")
else:
    bad_day_idata = sample_model(
        bad_day_model,
        draws=DRAWS,
        tune=TUNE,
        chains=CHAINS,
        target_accept=TARGET_ACCEPT,
    )
    save_inference_data(bad_day_idata, bad_day_cache_path)
    print(f"Saved -2% bad-day posterior samples to {bad_day_cache_path}")

bad_day_idata

### Posterior bad-day probability and expected bad days per year

The Bernoulli model estimates each stock's probability of a bad day. Multiplying by 252 trading days converts the probability into an expected number of bad days per year.

In [ ]:
bad_day_summary = posterior_bad_day_summary(bad_day_idata, bad_day_symbol_lookup)
bad_day_summary = bad_day_summary.sort_values(
    "mean_bad_day_probability", ascending=False
).reset_index(drop=True)

styled_bad_day_summary = bad_day_summary.style.format(
    {
        "mean_bad_day_probability": "{:.2%}",
        "bad_day_probability_q5": "{:.2%}",
        "bad_day_probability_q95": "{:.2%}",
        "expected_bad_days_per_year": "{:.1f}",
    }
)

bad_day_summary_path = save_summary_table(
    bad_day_summary,
    TABLES_DIR / "05_downside_model_2pct_summary.csv",
)
print(f"Saved -2% downside summary to {bad_day_summary_path}")
display(styled_bad_day_summary)

In [ ]:
bad_day_plot_df = bad_day_summary.sort_values("mean_bad_day_probability")

fig, ax = plt.subplots(figsize=(10, 6))
ax.errorbar(
    bad_day_plot_df["mean_bad_day_probability"],
    bad_day_plot_df["symbol"],
    xerr=[
        bad_day_plot_df["mean_bad_day_probability"] - bad_day_plot_df["bad_day_probability_q5"],
        bad_day_plot_df["bad_day_probability_q95"] - bad_day_plot_df["mean_bad_day_probability"],
    ],
    fmt="o",
    color="#E45756",
    ecolor="#E45756",
    elinewidth=2,
    capsize=4,
)
ax.set_title("Posterior probability of a -2% or worse day")
ax.set_xlabel("Daily probability")
ax.set_ylabel("Symbol")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.1%}"))
fig.tight_layout()
bad_day_probability_path = FIGURES_DIR / "05_posterior_bad_day_probability_minus_2pct.png"
fig.savefig(bad_day_probability_path, dpi=300, bbox_inches="tight")
bad_day_probability_path

### Probability each stock has the highest downside risk

The next table ranks stocks by posterior probability of having the highest -2% bad-day probability.

In [ ]:
highest_bad_day_risk = posterior_probability_highest_bad_day_risk(
    bad_day_idata,
    bad_day_symbol_lookup,
).sort_values("probability_highest_bad_day_risk", ascending=False)

highest_bad_day_risk_path_csv = save_summary_table(
    highest_bad_day_risk,
    TABLES_DIR / "05_downside_model_2pct_highest_risk_probability.csv",
)
print(f"Saved highest -2% downside risk probabilities to {highest_bad_day_risk_path_csv}")

display(
    highest_bad_day_risk.style.format(
        {"probability_highest_bad_day_risk": "{:.1%}"}
    )
)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(
    data=highest_bad_day_risk.sort_values("probability_highest_bad_day_risk"),
    x="probability_highest_bad_day_risk",
    y="symbol",
    color="#B279A2",
    ax=ax,
)
ax.set_xlim(0, max(1.0, highest_bad_day_risk["probability_highest_bad_day_risk"].max() * 1.1))
ax.set_title("Posterior probability of highest -2% downside risk")
ax.set_xlabel("Pr(highest -2% bad-day probability)")
ax.set_ylabel("Symbol")
fig.tight_layout()
highest_bad_day_risk_path = FIGURES_DIR / "05_probability_highest_bad_day_risk_minus_2pct.png"
fig.savefig(highest_bad_day_risk_path, dpi=300, bbox_inches="tight")
highest_bad_day_risk_path

## 5. Part C: Extreme-loss sensitivity

The -2% threshold captures relatively common bad days. A -5% threshold captures rarer **extreme-loss days**:

$$\text{simple_return} \leq -5\%$$

Because extreme losses are rare, posterior intervals are often wider and hierarchical pooling matters more.

In [ ]:
extreme_loss_threshold = -0.05
extreme_y, extreme_stock_idx, extreme_symbol_lookup, extreme_modeling_df = prepare_downside_data(
    daily_returns,
    threshold=extreme_loss_threshold,
)
extreme_symbols = [extreme_symbol_lookup[idx] for idx in sorted(extreme_symbol_lookup)]

extreme_cache_path = POSTERIOR_DIR / "downside_model_5pct.nc"
extreme_loss_model = build_bayesian_bad_day_model(
    y=extreme_y,
    stock_idx=extreme_stock_idx,
    n_stocks=len(extreme_symbols),
)

if extreme_cache_path.exists() and not REFIT_MODELS:
    extreme_idata = load_inference_data(extreme_cache_path)
    print(f"Loaded cached -5% extreme-loss posterior samples from {extreme_cache_path}")
else:
    extreme_idata = sample_model(
        extreme_loss_model,
        draws=DRAWS,
        tune=TUNE,
        chains=CHAINS,
        target_accept=TARGET_ACCEPT,
    )
    save_inference_data(extreme_idata, extreme_cache_path)
    print(f"Saved -5% extreme-loss posterior samples to {extreme_cache_path}")

extreme_idata

In [ ]:
extreme_summary = posterior_bad_day_summary(extreme_idata, extreme_symbol_lookup)
extreme_summary = extreme_summary.rename(
    columns={
        "mean_bad_day_probability": "mean_extreme_loss_probability",
        "bad_day_probability_q5": "extreme_loss_probability_q5",
        "bad_day_probability_q95": "extreme_loss_probability_q95",
        "expected_bad_days_per_year": "expected_extreme_loss_days_per_year",
    }
)

threshold_comparison = (
    bad_day_summary[
        [
            "symbol",
            "mean_bad_day_probability",
            "bad_day_probability_q5",
            "bad_day_probability_q95",
            "expected_bad_days_per_year",
        ]
    ]
    .merge(extreme_summary, on="symbol")
    .sort_values("mean_bad_day_probability", ascending=False)
    .reset_index(drop=True)
)

styled_threshold_comparison = threshold_comparison.style.format(
    {
        "mean_bad_day_probability": "{:.2%}",
        "bad_day_probability_q5": "{:.2%}",
        "bad_day_probability_q95": "{:.2%}",
        "expected_bad_days_per_year": "{:.1f}",
        "mean_extreme_loss_probability": "{:.2%}",
        "extreme_loss_probability_q5": "{:.2%}",
        "extreme_loss_probability_q95": "{:.2%}",
        "expected_extreme_loss_days_per_year": "{:.1f}",
    }
)

extreme_summary_path = save_summary_table(
    extreme_summary,
    TABLES_DIR / "05_downside_model_5pct_summary.csv",
)
threshold_comparison_path = save_summary_table(
    threshold_comparison,
    TABLES_DIR / "05_downside_threshold_comparison.csv",
)
print(f"Saved -5% downside summary to {extreme_summary_path}")
print(f"Saved downside threshold comparison to {threshold_comparison_path}")
display(styled_threshold_comparison)

In [ ]:
threshold_plot_df = threshold_comparison.melt(
    id_vars="symbol",
    value_vars=["mean_bad_day_probability", "mean_extreme_loss_probability"],
    var_name="risk_threshold",
    value_name="posterior_daily_probability",
)
threshold_plot_df["risk_threshold"] = threshold_plot_df["risk_threshold"].map(
    {
        "mean_bad_day_probability": "Bad day (<= -2%)",
        "mean_extreme_loss_probability": "Extreme loss (<= -5%)",
    }
)

fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(
    data=threshold_plot_df,
    x="posterior_daily_probability",
    y="symbol",
    hue="risk_threshold",
    ax=ax,
)
ax.set_title("Bayesian downside-risk sensitivity to the loss threshold")
ax.set_xlabel("Posterior daily probability")
ax.set_ylabel("Symbol")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.1%}"))
ax.legend(title="Threshold")
fig.tight_layout()
threshold_comparison_path = FIGURES_DIR / "05_downside_threshold_sensitivity.png"
fig.savefig(threshold_comparison_path, dpi=300, bbox_inches="tight")
threshold_comparison_path

## 6. Compare Bayesian estimates to raw historical frequencies

Raw frequencies are useful descriptive statistics, but they do not show uncertainty and can be noisy when events are rare. The Bayesian estimates below are partially pooled across stocks, which can stabilize estimates while still allowing stock-level differences.

In [ ]:
bayesian_vs_raw = (
    volatility_summary[
        [
            "symbol",
            "posterior_annualized_volatility_mean",
            "annualized_volatility_ci_5",
            "annualized_volatility_ci_95",
        ]
    ]
    .merge(
        bad_day_summary[
            [
                "symbol",
                "mean_bad_day_probability",
                "bad_day_probability_q5",
                "bad_day_probability_q95",
            ]
        ],
        on="symbol",
    )
    .merge(
        extreme_summary[
            [
                "symbol",
                "mean_extreme_loss_probability",
                "extreme_loss_probability_q5",
                "extreme_loss_probability_q95",
            ]
        ],
        on="symbol",
    )
    .merge(
        raw_risk_summary[
            [
                "symbol",
                "raw_annualized_volatility",
                "raw_bad_day_frequency_2pct",
                "raw_extreme_loss_frequency_5pct",
            ]
        ],
        on="symbol",
    )
    .sort_values("posterior_annualized_volatility_mean", ascending=False)
    .reset_index(drop=True)
)

bayesian_vs_raw_path = save_summary_table(
    bayesian_vs_raw,
    TABLES_DIR / "05_bayesian_vs_raw_risk_summary.csv",
)
print(f"Saved Bayesian-vs-raw risk summary to {bayesian_vs_raw_path}")

display(
    bayesian_vs_raw.style.format(
        {
            "posterior_annualized_volatility_mean": "{:.2%}",
            "annualized_volatility_ci_5": "{:.2%}",
            "annualized_volatility_ci_95": "{:.2%}",
            "raw_annualized_volatility": "{:.2%}",
            "mean_bad_day_probability": "{:.2%}",
            "bad_day_probability_q5": "{:.2%}",
            "bad_day_probability_q95": "{:.2%}",
            "raw_bad_day_frequency_2pct": "{:.2%}",
            "mean_extreme_loss_probability": "{:.2%}",
            "extreme_loss_probability_q5": "{:.2%}",
            "extreme_loss_probability_q95": "{:.2%}",
            "raw_extreme_loss_frequency_5pct": "{:.2%}",
        }
    )
)

In [ ]:
comparison_plot_df = bayesian_vs_raw.melt(
    id_vars="symbol",
    value_vars=[
        "mean_bad_day_probability",
        "raw_bad_day_frequency_2pct",
        "mean_extreme_loss_probability",
        "raw_extreme_loss_frequency_5pct",
    ],
    var_name="estimate",
    value_name="daily_probability",
)
comparison_plot_df["estimate"] = comparison_plot_df["estimate"].map(
    {
        "mean_bad_day_probability": "Bayesian <= -2%",
        "raw_bad_day_frequency_2pct": "Raw <= -2%",
        "mean_extreme_loss_probability": "Bayesian <= -5%",
        "raw_extreme_loss_frequency_5pct": "Raw <= -5%",
    }
)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(
    data=comparison_plot_df,
    x="daily_probability",
    y="symbol",
    hue="estimate",
    ax=ax,
)
ax.set_title("Bayesian downside probabilities versus raw historical frequencies")
ax.set_xlabel("Daily event probability")
ax.set_ylabel("Symbol")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.1%}"))
ax.legend(title="Estimate", ncols=2)
fig.tight_layout()
bayesian_vs_raw_path = FIGURES_DIR / "05_bayesian_vs_raw_downside_frequencies.png"
fig.savefig(bayesian_vs_raw_path, dpi=300, bbox_inches="tight")
bayesian_vs_raw_path

## 7. Conclusion

- **Volatility and downside risk are related but not identical.** Volatility summarizes typical return dispersion, while downside probabilities focus on a specific loss threshold that investors and risk managers may care about directly.
- **The Bayesian approach shows uncertainty around risk estimates.** Posterior intervals make clear when two stocks have meaningfully different risk profiles and when their estimates overlap.
- **High historical risk does not always mean a precise risk estimate.** Rare -5% events can produce unstable raw frequencies; Bayesian partial pooling helps temper extreme estimates and communicates uncertainty.
- **Practical takeaway:** use annualized volatility for broad risk budgeting, but pair it with threshold-based downside probabilities when the question is about the chance of a painful one-day loss.

## 8. Saved figure paths

The key visual outputs are saved under `reports/figures` for reuse in reports and dashboards.

In [ ]:
saved_figures = pd.DataFrame(
    {
        "figure": [
            "Posterior annualized volatility by stock",
            "Probability of highest volatility",
            "Posterior -2% bad-day probability",
            "Probability of highest -2% downside risk",
            "Downside threshold sensitivity",
            "Bayesian versus raw downside frequencies",
        ],
        "path": [
            volatility_plot_path,
            highest_volatility_path,
            bad_day_probability_path,
            highest_bad_day_risk_path,
            threshold_comparison_path,
            bayesian_vs_raw_path,
        ],
    }
)
display(saved_figures)